# EDGE AI Battery Monitoring — SOC Estimation (Comparative Study)

**Goal:** Estimate live State of Charge (SOC, 0-100%) from voltage/current/temperature, using 3 models suited for eventual ESP32-S3 deployment via TFLite Micro.

**Models compared:**
1. Random Forest (classical ML baseline, windowed statistical features)
2. MLP / Feedforward Neural Network (flattened window, TFLite Micro friendly)
3. LSTM (sequence model, best fit for SOC's inherently time-dependent nature)

**Data:** `timeseries_soc_FINAL.csv` — coulomb-counted SOC ground truth per timestep, extracted from NASA `.mat` battery cycling data (32 batteries, cleaned).

**Split strategy:** by *battery*, not by cycle — some whole batteries are held out for testing, so the models are evaluated on batteries never seen in training. This is stricter (and more honest) than a random row-level split, which would leak information between cycles of the same battery.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)


## 1. Load data

In [ ]:
df = pd.read_csv('timeseries_soc_FINAL.csv')
print(df.shape)
print(df['battery_id'].nunique(), 'batteries')
df.head()


## 2. Resample to a uniform time grid

NASA's logger sampled at an irregular interval (~9-14s median, varies by battery). Your ESP32-S3 will sample at a fixed rate in deployment, so we resample every cycle onto a uniform grid (default: every 10s) via linear interpolation, before building windows.


In [ ]:
def resample_cycle(g, dt=10.0):
    g = g.sort_values('t_s')
    t_max = g['t_s'].max()
    new_t = np.arange(0, t_max, dt)
    if len(new_t) < 5:
        return None
    out = pd.DataFrame({'t_s': new_t})
    for col in ['voltage', 'current', 'temperature', 'soc']:
        out[col] = np.interp(new_t, g['t_s'], g[col])
    out['battery_id'] = g['battery_id'].iloc[0]
    out['cycle'] = g['cycle'].iloc[0]
    return out

resampled = []
for (b, c), g in df.groupby(['battery_id', 'cycle']):
    r = resample_cycle(g)
    if r is not None:
        resampled.append(r)

res_df = pd.concat(resampled, ignore_index=True)
print('Resampled shape:', res_df.shape)
res_df.to_csv('resampled_soc.csv', index=False)


## 3. Battery-level train/test split

Holding out entire batteries (not just cycles) for testing gives an honest estimate of how the model generalizes to a battery it has never seen — the real deployment scenario.


In [ ]:
batteries = sorted(res_df['battery_id'].unique())
print(len(batteries), 'batteries total')

rng = np.random.RandomState(42)
shuffled = rng.permutation(batteries)
n_test = max(1, int(len(batteries) * 0.2))
test_batteries = set(shuffled[:n_test])
train_batteries = set(shuffled[n_test:])

print('Test batteries (held out):', sorted(test_batteries))
print('Train batteries:', len(train_batteries))


## 4. Build sliding windows

Each model sees a **window** of the last `WINDOW` timesteps (voltage, current, temperature) and predicts the SOC at the current timestep. `WINDOW=12` with a 10s grid = 120 seconds of lookback — adjust based on your ESP32's sampling rate and available RAM.


In [ ]:
WINDOW = 12  # number of past timesteps in each input window

def make_windows(sub_df, window=WINDOW):
    X, y = [], []
    for (b, c), g in sub_df.groupby(['battery_id', 'cycle']):
        g = g.sort_values('t_s')
        v = g['voltage'].values
        i = g['current'].values
        t = g['temperature'].values
        soc = g['soc'].values
        for k in range(window, len(g)):
            X.append(np.stack([v[k-window:k], i[k-window:k], t[k-window:k]], axis=1))
            y.append(soc[k])
    return np.array(X), np.array(y)

train_df = res_df[res_df['battery_id'].isin(train_batteries)]
test_df = res_df[res_df['battery_id'].isin(test_batteries)]

print('Building training windows...')
X_train, y_train = make_windows(train_df)
print('Building test windows...')
X_test, y_test = make_windows(test_df)

print('X_train:', X_train.shape, ' y_train:', y_train.shape)
print('X_test:', X_test.shape, ' y_test:', y_test.shape)


## 5. Normalize inputs

Fit the scaler on **training data only** to avoid leaking test-set statistics.


In [ ]:
mean = X_train.reshape(-1, 3).mean(axis=0)
std = X_train.reshape(-1, 3).std(axis=0)

X_train_n = (X_train - mean) / std
X_test_n = (X_test - mean) / std

print('Feature means (V, I, T):', mean)
print('Feature stds  (V, I, T):', std)


## 6. Model 1 — Random Forest (classical ML baseline)

Uses engineered per-window statistics (mean/min/max/last value/delta) rather than the raw sequence, since tree ensembles don't natively exploit sequence order.


In [ ]:
def flatten_stats(X):
    feats = []
    for ch in range(X.shape[2]):
        c = X[:, :, ch]
        feats += [c.mean(axis=1), c.min(axis=1), c.max(axis=1), c[:, -1], c[:, -1] - c[:, 0]]
    return np.stack(feats, axis=1)

X_train_feat = flatten_stats(X_train_n)
X_test_feat = flatten_stats(X_test_n)

rf = RandomForestRegressor(n_estimators=200, max_depth=14, n_jobs=-1, random_state=42)
rf.fit(X_train_feat, y_train)

rf_pred = rf.predict(X_test_feat)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)
print(f'Random Forest -> MAE: {rf_mae:.3f}  RMSE: {rf_rmse:.3f}  R2: {rf_r2:.4f}')


## 7. Model 2 — MLP (Feedforward Neural Network)

Flattens the window into a single vector. Simple, fast, and directly convertible to TFLite Micro for ESP32-S3.


In [ ]:
X_train_flat = X_train_n.reshape(len(X_train_n), -1)
X_test_flat = X_test_n.reshape(len(X_test_n), -1)

mlp = keras.Sequential([
    keras.layers.Input(shape=(X_train_flat.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1)
])
mlp.compile(optimizer='adam', loss='mse')

history_mlp = mlp.fit(
    X_train_flat, y_train,
    validation_split=0.1,
    epochs=30, batch_size=512, verbose=1
)

mlp_pred = mlp.predict(X_test_flat, verbose=0).flatten()
mlp_mae = mean_absolute_error(y_test, mlp_pred)
mlp_rmse = np.sqrt(mean_squared_error(y_test, mlp_pred))
mlp_r2 = r2_score(y_test, mlp_pred)
print(f'MLP -> MAE: {mlp_mae:.3f}  RMSE: {mlp_rmse:.3f}  R2: {mlp_r2:.4f}')


## 8. Model 3 — LSTM (sequence model)

Takes the raw (voltage, current, temperature) sequence directly, without manual feature engineering — best suited to SOC since it's inherently a time-dependent quantity.


In [ ]:
lstm = keras.Sequential([
    keras.layers.Input(shape=(X_train_n.shape[1], X_train_n.shape[2])),
    keras.layers.LSTM(32),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1)
])
lstm.compile(optimizer='adam', loss='mse')

history_lstm = lstm.fit(
    X_train_n, y_train,
    validation_split=0.1,
    epochs=30, batch_size=512, verbose=1
)

lstm_pred = lstm.predict(X_test_n, verbose=0).flatten()
lstm_mae = mean_absolute_error(y_test, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(y_test, lstm_pred))
lstm_r2 = r2_score(y_test, lstm_pred)
print(f'LSTM -> MAE: {lstm_mae:.3f}  RMSE: {lstm_rmse:.3f}  R2: {lstm_r2:.4f}')


## 9. Comparative results

In [ ]:
results = pd.DataFrame({
    'Model': ['Random Forest', 'MLP', 'LSTM'],
    'MAE (%)': [rf_mae, mlp_mae, lstm_mae],
    'RMSE (%)': [rf_rmse, mlp_rmse, lstm_rmse],
    'R2': [rf_r2, mlp_r2, lstm_r2],
    'Params/Nodes': [rf.n_estimators, mlp.count_params(), lstm.count_params()]
})
results


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pred) in zip(axes, [('Random Forest', rf_pred), ('MLP', mlp_pred), ('LSTM', lstm_pred)]):
    ax.scatter(y_test, pred, s=2, alpha=0.3)
    ax.plot([0, 100], [0, 100], 'r--')
    ax.set_xlabel('True SOC (%)')
    ax.set_ylabel('Predicted SOC (%)')
    ax.set_title(name)
plt.tight_layout()
plt.savefig('soc_model_comparison.png', dpi=120)
plt.show()


## 10. Edge deployment notes (ESP32-S3)

- **MLP and LSTM** convert directly to TFLite Micro (`tf.lite.TFLiteConverter`), then to a `.tflite` file, then to a C byte array for flashing.
- **Random Forest** doesn't have a native TFLite path — export it as C code instead (e.g. via `emlearn` or `m2cgen`), or use it only as your offline comparative baseline rather than the deployed model.
- Apply **post-training int8 quantization** on the MLP/LSTM before conversion — cuts model size ~4x with usually small accuracy loss, and ESP32-S3 has hardware acceleration for int8 ops.
- Keep `WINDOW` small (12-20 steps) to limit the RAM needed to buffer the input sequence on-device.
- Next notebook: reuse this same pattern (windowing, battery-level split, 3-model comparison) for **SOH** and **RUL**, using the cycle-level summary CSV instead of the timeseries one.
